# Custom Field Serialization (`@field_serializer`)

While Pydantic does an excellent job of serializing data by default (e.g., automatically converting `datetime` objects to ISO-8601 strings for JSON), there are times when you need strict control over the output. For instance, you might want to format a date a specific way, round floats to two decimal places, or enforce UTC timezones before outputting data.

To do this, Pydantic provides the `@field_serializer` decorator.

## 1. Basic Usage of `@field_serializer`

The `@field_serializer` is a decorator applied to an **instance method**. Because it runs *after* the model is fully validated and instantiated, the method must accept `self` and the `value` of the field being serialized.

```python
from pydantic import BaseModel, field_serializer
from datetime import datetime

class Meeting(BaseModel):
    meeting_time: datetime

    # The decorator specifies which field this serializer applies to
    @field_serializer('meeting_time')
    def serialize_meeting_time(self, value: datetime) -> str:
        # Returns a custom formatted string: "2020/01/01 12:00 PM"
        return value.strftime("%Y/%m/%d %I:%M %p")

m = Meeting(meeting_time="2020-01-01T12:00:00")

# The serializer intercepts the output!
print(m.model_dump()) 
# {'meeting_time': '2020/01/01 12:00 PM'}

```

## 2. Handling Nulls: The `when_used` Argument

By default, your custom serializer runs **always**—even if the field's value is `None`. If you try to run `.strftime()` on `None`, your application will crash.

To prevent having to write `if value is None:` in every custom serializer, Pydantic provides the `when_used` argument.

* `'always'` (Default): Runs for every serialization.
* `'unless-none'`: Runs *only* if the value is not `None`. (Pydantic will automatically output `None`/`null` for you).
* `'json'`: Runs *only* during `model_dump_json()`.
* `'json-unless-none'`: Runs *only* during `model_dump_json()`, provided the value is not `None`.

```python
class OptionalMeeting(BaseModel):
    meeting_time: datetime | None = None

    @field_serializer('meeting_time', when_used='unless-none')
    def serialize_meeting_time(self, value: datetime):
        return value.strftime("%Y/%m/%d %I:%M %p")

m1 = OptionalMeeting(meeting_time=None)
print(m1.model_dump()) # {'meeting_time': None} -> Custom logic was safely skipped!

```

## 3. Differentiating Output (`FieldSerializationInfo`)

Sometimes, you want to return a native Python object (like an aware `datetime` object) when serializing to a Python Dictionary, but you want to return a specific formatted String when serializing to JSON.

You can ask Pydantic to inject a third argument into your serializer: `FieldSerializationInfo`. You can use the `.mode_is_json()` method on this object to branch your logic.

```python
from pydantic import BaseModel, field_serializer, FieldSerializationInfo
from datetime import datetime

class DualModeModel(BaseModel):
    dt: datetime

    @field_serializer('dt', when_used='unless-none')
    def serialize_dt(self, value: datetime, info: FieldSerializationInfo):
        
        # Branch 1: model_dump_json() was called
        if info.mode_is_json():
            return value.strftime("%Y/%m/%d %I:%M %p")
        
        # Branch 2: model_dump() was called
        return value # Leave it as a native Python datetime object

```

## 4. Advanced Example: Enforcing UTC on Output

A very common real-world use case is ensuring that no matter what timezone a user inputs, the database and JSON endpoints only ever receive UTC time.

Here is how you can use `@field_serializer` alongside `pytz` to ensure all output is UTC-aware.

```python
import pytz
from datetime import datetime
from pydantic import BaseModel, field_serializer, FieldSerializationInfo

class Event(BaseModel):
    timestamp: datetime
    
    @field_serializer('timestamp', when_used='unless-none')
    def enforce_utc(self, dt: datetime, info: FieldSerializationInfo):
        # 1. Standardize to UTC
        if dt.tzinfo is None:
            # If naive, assume it's UTC and localize it
            dt_utc = pytz.utc.localize(dt)
        else:
            # If aware, convert it to UTC
            dt_utc = dt.astimezone(pytz.utc)
            
        # 2. Format based on output type
        if info.mode_is_json():
            # Standard ISO format ending with 'Z' for JSON
            return dt_utc.strftime("%Y-%m-%dT%H:%M:%SZ")
        
        # Return Python datetime object for dict dumps
        return dt_utc 

# Test with US Eastern Time (UTC-5)
eastern = pytz.timezone('US/Eastern')
local_time = eastern.localize(datetime(2021, 1, 1, 0, 0, 0))

e = Event(timestamp=local_time)

# JSON Dump shifts the time forward 5 hours to UTC and formats with 'Z'
print(e.model_dump_json())
# {"timestamp": "2021-01-01T05:00:00Z"}

```

### Interview Preparation: Field Serializers

<details>
<summary><b>1. You wrote a `@field_serializer` to format a `datetime` object, but your application crashes with an `AttributeError` when the field is left empty. Why did this happen and what is the cleanest Pydantic way to fix it?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
This happened because the field is nullable, and by default, Pydantic's `@field_serializer` runs even if the value is `None`. The custom function attempted to call a method like `.strftime()` on a `NoneType` object.<br><br>
While you could write an `if value is None: return None` check inside the function, the cleanest Pydantic way is to pass `when_used='unless-none'` to the `@field_serializer` decorator. This tells Pydantic to skip the custom logic automatically if the value is `None`.
</details>

<details>
<summary><b>2. In a `@field_serializer` function, what are the required arguments in the method signature?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
Because the serializer is an instance method, it must accept `self` as the first argument, and the `value` of the field being serialized as the second argument.<br><br>
Optionally, you can add a third argument typed as `FieldSerializationInfo` if you need to know context about the serialization (e.g., whether it is serializing to JSON or a dictionary).
</details>

<details>
<summary><b>3. You need a field to output as a native `Decimal` object when calling `model_dump()`, but it must be output as a rounded `float` when calling `model_dump_json()`. How can you achieve this?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
You achieve this by using the `FieldSerializationInfo` object inside a `@field_serializer`. <br><br>
You add a third argument to your serializer method typed as `FieldSerializationInfo`. Inside the method, you call `info.mode_is_json()`. If it returns `True`, you return `round(float(value), 2)`. If it returns `False` (meaning it's a standard dictionary dump), you simply return the original `Decimal` value.
</details>

<details>
<summary><b>4. True or False: The `@field_serializer` can be used to modify data <i>before</i> it gets validated and saved to the Pydantic model instance. Explain your reasoning.</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
False. <br><br>
The `@field_serializer` is strictly an *output* mechanism. It intercepts the data on its way out of the model during `model_dump()` or `model_dump_json()`. It does not modify the data stored inside the Python instance itself. To modify data *before* or *during* validation (on input), you would use a `@field_validator` instead.
</details>